# ML Zoomcamp 2026 — Homework 3 (Classification)
**Nicolau Calado Jofilsan** · 2026 cohort (DataTalks.Club)

**EN.** Lead scoring: predict whether a lead signed up for the platform (`converted`).
Dataset: the pinned 2026 release `course_lead_scoring_2026.csv` from the course repository
(`cohorts/2026/data/`). Deadline: Mon 12/10/2026, 20:00 (account timezone).

**PT.** Classificação de leads: prever se o lead se inscreveu na plataforma (`converted`).
Base: versão fixada de 2026, `course_lead_scoring_2026.csv`, do repositório do curso
(`cohorts/2026/data/`). Prazo: seg 12/10/2026, 20:00 (horário da conta).

Topics / Conteúdo: missing values, mode, correlation matrix, mutual information,
one-hot encoding with `DictVectorizer`, logistic regression, feature elimination, regularization.

## 0. Setup and data / Preparação e dados

In [1]:
import os
import urllib.request

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score, accuracy_score

pd.set_option("display.max_columns", 50)

DATA_FILE = "course_lead_scoring_2026.csv"
DATA_URL = (
    "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/"
    "main/cohorts/2026/data/course_lead_scoring_2026.csv"
)

if not os.path.exists(DATA_FILE):
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

df = pd.read_csv(DATA_FILE)
print("shape:", df.shape)
df.head()

shape: (5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## 1. Data preparation / Preparação dos dados

**EN.** Missing values: `'NA'` for categorical features, `0.0` for numerical ones, as the homework asks.
**PT.** Valores faltantes: `'NA'` nas categóricas e `0.0` nas numéricas, conforme o enunciado.

In [2]:
print(df.isnull().sum())

categorical = ["lead_source", "industry", "employment_status", "location"]
numerical = ["annual_income", "number_of_courses_viewed", "interaction_count", "lead_score"]
target = "converted"

for c in categorical:
    df[c] = df[c].fillna("NA")

for c in numerical:
    df[c] = df[c].fillna(0.0)

print("\nmissing values after filling:", df.isnull().sum().sum())

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

missing values after filling: 0


## Q1. Mode of `industry` / Moda da coluna `industry`

In [3]:
print(df["industry"].value_counts())
print("\nQ1 =", df["industry"].mode()[0])

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

Q1 = technology


**Answer / Resposta Q1: `technology`** — 1173 rows, ahead of `retail` (925) and `healthcare` (840).
The 240 filled `NA` values do not change the mode / os 240 valores preenchidos como `NA` não alteram a moda.

## Q2. Correlation matrix / Matriz de correlação

In [4]:
corr = df[numerical].corr()
display(corr.round(4))

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
for a, b in pairs:
    print(f"{a} x {b}: {corr.loc[a, b]:.4f}")

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.0000,0.1613,0.1228,0.2295
number_of_courses_viewed,0.1613,1.0000,0.7216,0.7572
interaction_count,0.1228,0.7216,1.0000,0.9157
lead_score,0.2295,0.7572,0.9157,1.0000


interaction_count x lead_score: 0.9157
number_of_courses_viewed x lead_score: 0.7572
number_of_courses_viewed x interaction_count: 0.7216
annual_income x interaction_count: 0.1228


**Answer / Resposta Q2: `interaction_count` and `lead_score`** — 0.9157

EN: the other listed pairs are lower (0.7572, 0.7216 and 0.1228). A correlation this high means the two
variables carry almost the same information, which matters later: with one-hot encoded features and no
scaling, strongly correlated inputs make individual coefficients unstable, even when accuracy does not move.

PT: os outros pares listados ficam abaixo (0,7572, 0,7216 e 0,1228). Uma correlação tão alta indica que as
duas variáveis carregam quase a mesma informação, o que importa adiante: sem padronização, variáveis muito
correlacionadas tornam os coeficientes individuais instáveis, ainda que a acurácia não mude.

## Split the data / Divisão dos dados

**EN.** 60/20/20 using the exact calls from the homework: first 80/20, then 75/25 of the first part.
**PT.** 60/20/20 com as chamadas exatas do enunciado: primeiro 80/20, depois 75/25 da primeira parte.

In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

# the target must not stay among the features / o alvo não pode ficar entre as variáveis
X_train_df = df_train.drop(columns=[target])
X_val_df = df_val.drop(columns=[target])
X_test_df = df_test.drop(columns=[target])

print("train:", len(df_train), "| val:", len(df_val), "| test:", len(df_test))
print("conversion rate in train / taxa de conversão no treino:", round(y_train.mean(), 4))

train: 3000 | val: 1000 | test: 1000
conversion rate in train / taxa de conversão no treino: 0.562


## Q3. Mutual information / Informação mútua

**EN.** Mutual information measures how much knowing one variable reduces uncertainty about the target.
Unlike correlation, it captures non-linear relationships and works with categorical variables.
**PT.** A informação mútua mede quanto conhecer uma variável reduz a incerteza sobre o alvo. Diferente da
correlação, capta relações não lineares e funciona com variáveis categóricas.

In [6]:
mi_scores = {}
for c in categorical:
    score = mutual_info_score(df_train[c], y_train)
    mi_scores[c] = round(score, 2)
    print(f"{c:>20}: {score:.6f}  ->  {round(score, 2)}")

print("\nQ3 =", max(mi_scores, key=mi_scores.get))

         lead_source: 0.034346  ->  0.03
            industry: 0.001995  ->  0.0
   employment_status: 0.018966  ->  0.02
            location: 0.001043  ->  0.0

Q3 = lead_source


**Answer / Resposta Q3: `lead_source`** — 0.03, ahead of `employment_status` (0.02);
`industry` and `location` round to 0.00.

EN: where the lead came from says more about conversion than who the lead is.
PT: de onde veio o lead diz mais sobre a conversão do que quem é o lead.

## Q4. Logistic regression / Regressão logística

**EN.** One-hot encoding with `DictVectorizer`, fitted on the training set only and reused on validation,
so both matrices have the same columns in the same order.
**PT.** One-hot encoding com `DictVectorizer`, ajustado só no treino e reaplicado na validação, para que as
duas matrizes tenham as mesmas colunas na mesma ordem.

In [7]:
features = categorical + numerical


def train_and_evaluate(feature_list, C=1.0):
    """Fit logistic regression on the training set, return validation accuracy."""
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(X_train_df[feature_list].to_dict(orient="records"))
    X_val = dv.transform(X_val_df[feature_list].to_dict(orient="records"))

    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    return accuracy_score(y_val, model.predict(X_val))


accuracy = train_and_evaluate(features)
print("validation accuracy:", accuracy)
print("Q4 =", round(accuracy, 2))

validation accuracy: 0.645
Q4 = 0.65


**Answer / Resposta Q4: 0.65** — 0.645 before rounding.

EN: a useful reference point: always predicting the majority class would give 0.597 on this validation set, so the model adds about 4.8 percentage points over that baseline.
PT: uma referência útil: prever sempre a classe majoritária daria 0,597 nesta validação, então o modelo acrescenta cerca de 4,8 pontos percentuais sobre essa linha de base.

In [8]:
baseline = max(y_val.mean(), 1 - y_val.mean())
print("majority-class baseline / linha de base da classe majoritária:", round(baseline, 3))

majority-class baseline / linha de base da classe majoritária: 0.597


## Q5. Feature elimination / Eliminação de variáveis

**EN.** Train without each feature and compare with the full-model accuracy. A small difference means the
feature adds little.
**PT.** Treinar sem cada variável e comparar com a acurácia do modelo completo. Uma diferença pequena
significa que a variável acrescenta pouco.

In [9]:
differences = {}
for f in features:
    acc_without = train_and_evaluate([x for x in features if x != f])
    differences[f] = accuracy - acc_without
    print(f"without {f:>26}: accuracy = {acc_without:.4f}   difference = {accuracy - acc_without:+.4f}")

options = ["lead_source", "number_of_courses_viewed", "interaction_count"]
print("\namong the options / entre as opções:")
for f in options:
    print(f"  {f:>26}: {differences[f]:+.4f}")
print("\nQ5 =", min(options, key=lambda f: differences[f]))

without                lead_source: accuracy = 0.6420   difference = +0.0030
without                   industry: accuracy = 0.6450   difference = +0.0000
without          employment_status: accuracy = 0.6450   difference = +0.0000


without                   location: accuracy = 0.6450   difference = +0.0000


without              annual_income: accuracy = 0.7240   difference = -0.0790


without   number_of_courses_viewed: accuracy = 0.6430   difference = +0.0020


without          interaction_count: accuracy = 0.6010   difference = +0.0440
without                 lead_score: accuracy = 0.6440   difference = +0.0010

among the options / entre as opções:
                 lead_source: +0.0030
    number_of_courses_viewed: +0.0020
           interaction_count: +0.0440

Q5 = number_of_courses_viewed


**Answer / Resposta Q5: `number_of_courses_viewed`** — difference of +0.002, below `lead_source`
(+0.003) and far below `interaction_count` (+0.044). It is also the smallest in absolute value, so the answer
holds under either reading of "smallest difference".

EN: one result outside the options deserves attention — removing `annual_income` *raises* accuracy from
0.645 to 0.724. A feature that hurts the model this much is worth investigating before trusting the model.
PT: um resultado fora das opções merece atenção: remover `annual_income` *aumenta* a acurácia de 0,645 para
0,724. Uma variável que prejudica tanto o modelo merece investigação antes de confiar no modelo.

## Q6. Regularized logistic regression / Regressão logística regularizada

**EN.** In scikit-learn, `C` is the inverse of regularization strength: small `C` means strong
regularization, which shrinks the coefficients towards zero.
**PT.** No scikit-learn, `C` é o inverso da força de regularização: `C` pequeno significa regularização
forte, que encolhe os coeficientes na direção de zero.

In [10]:
scores_C = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    acc_C = train_and_evaluate(features, C=C)
    scores_C[C] = round(acc_C, 3)
    print(f"C = {C:<10}: accuracy = {acc_C:.4f}  ->  {round(acc_C, 3)}")

best = min(scores_C, key=lambda c: (-scores_C[c], c))  # best accuracy, smallest C on ties
print("\nQ6 =", best)

C = 1e-06     : accuracy = 0.5980  ->  0.598


C = 1e-05     : accuracy = 0.5980  ->  0.598
C = 0.0001    : accuracy = 0.6130  ->  0.613
C = 0.001     : accuracy = 0.6450  ->  0.645

Q6 = 0.001


**Answer / Resposta Q6: 0.001** — 0.645, against 0.613, 0.598 and 0.598.

EN: accuracy grows with `C` across the whole range, so the strongest regularization is the worst here. With
`C = 0.000001` the coefficients are crushed to near zero and the model is reduced to little more than the majority-class rule (0.598, against a baseline of 0.597). Among the values offered, 0.001 is the mildest regularization, and it recovers the
unregularized result of Q4.

PT: a acurácia cresce com `C` em toda a faixa, então a regularização mais forte é a pior aqui. Com
`C = 0,000001` os coeficientes são esmagados para perto de zero e o modelo fica reduzido a pouco mais que a regra da classe majoritária (0,598, contra uma linha de base de 0,597). Entre os valores oferecidos, 0,001 é a regularização mais branda, e
recupera o resultado sem regularização da Q4.

### Homework 3 — answers / respostas

| Question | Answer |
|---|---|
| Q1 Mode of `industry` | **`technology`** |
| Q2 Biggest correlation | **`interaction_count` and `lead_score`** (0.9157) |
| Q3 Biggest mutual information | **`lead_source`** (0.03) |
| Q4 Validation accuracy | **0.65** |
| Q5 Smallest difference | **`number_of_courses_viewed`** (+0.002) |
| Q6 Best `C` | **0.001** (accuracy 0.645) |

### Notes / Observações

**EN.** 1) `DictVectorizer` is fitted on the training set and only applied to validation, which keeps the
encoding out of the validation data. 2) Both splits use `random_state=42`, exactly as the homework specifies,
so the numbers are reproducible. 3) The dataset has no grouping column (each row is an independent lead), so
a random split is appropriate here.

**PT.** 1) O `DictVectorizer` é ajustado no treino e apenas aplicado na validação, o que mantém a codificação
fora dos dados de validação. 2) Os dois splits usam `random_state=42`, exatamente como o enunciado pede, então
os números são reprodutíveis. 3) A base não tem coluna de agrupamento (cada linha é um lead independente),
então o split aleatório é adequado aqui.